# CP3 — Continual Learning

Notebook ini mendemonstrasikan implementasi CP3 untuk project Chicago Risk Score API:
1. Simulasi data baru berbasis waktu (temporal split)
2. Deteksi drift menggunakan kenaikan relatif MAE
3. Retraining dengan checkpoint dan quality gate
4. Versioning model beserta konteks evaluasinya

**Cara menjalankan:** Klik `Runtime > Run All` dari root proyek yang sudah ada `app/` folder-nya.

Notebook ini memakai data processed yang sudah tersedia di repository dan memanggil modul `app.continual_learning`.

## 1. Persiapan Notebook dan Import Library

Bagian ini menyiapkan environment notebook, mengatur path kerja, dan mengimpor library yang diperlukan.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

# Pastikan root proyek ada di sys.path agar bisa import app.*
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.config import BASE_DIR, CP3_BATCH_COUNT, CP3_DRIFT_THRESHOLD, CP3_QUALITY_GATE, CP3_RANDOM_STATE
from app.continual_learning import (
    append_model_version,
    build_version_entry,
    detect_drift,
    load_event_history,
    read_model_versions,
    run_cp3_pipeline,
    split_time_batches,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

DATA_PATH = Path(BASE_DIR) / 'data' / 'processed' / 'features_labels_events.csv'
OUTPUT_DIR = Path(BASE_DIR) / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data path    : {DATA_PATH}')
print(f'Output dir   : {OUTPUT_DIR}')

## 2. Definisikan Konfigurasi dan Parameter CP3

Parameter berikut dipakai sepanjang notebook agar konsisten dengan implementasi produksi di `app/continual_learning.py`.

In [ ]:
cp3_config = {
    'batch_count': CP3_BATCH_COUNT,        # 4 batch temporal
    'drift_threshold': CP3_DRIFT_THRESHOLD, # MAE naik >15% = drift
    'quality_gate': CP3_QUALITY_GATE,      # kandidat harus lebih baik >=5%
    'random_state': CP3_RANDOM_STATE,
    'max_rows_demo': 150_000,              # cap untuk demo cepat
    'persist_outputs': False,              # True untuk benar-benar simpan model baru
}
print('Konfigurasi CP3:')
print(json.dumps(cp3_config, indent=2))

## 3. Siapkan Data Input

Notebook ini memakai data processed yang sudah tersedia. Kita juga buat mock input sederhana untuk dokumentasi alur.

In [ ]:
history = load_event_history(max_rows=cp3_config['max_rows_demo'])

mock_input = {
    'datetime': '2026-07-30T14:00:00',
    'lat': 41.86,
    'lon': -87.67,
    'batch_count': cp3_config['batch_count'],
    'drift_threshold': cp3_config['drift_threshold'],
}

print(f'History rows : {len(history):,}')
print(f'Date range   : {history["Date"].min().date()} s/d {history["Date"].max().date()}')
print('\nMock input:')
print(json.dumps(mock_input, indent=2))

## 4. Bangun Struktur Data — Temporal Batch Split

Data historis dibagi menjadi N batch berdasarkan urutan waktu. Batch terakhir berfungsi sebagai holdout tetap.

In [ ]:
batches = split_time_batches(history, batch_count=cp3_config['batch_count'])
model_versions_snapshot = read_model_versions()

batch_windows = [
    {
        'batch_index': index,
        'rows': len(batch),
        'start_date': batch['Date'].min().date().isoformat(),
        'end_date': batch['Date'].max().date().isoformat(),
        'is_holdout': index == len(batches) - 1,
    }
    for index, batch in enumerate(batches)
]

print(f'Jumlah batch : {len(batches)}')
print(f'Versionin awal: {len(model_versions_snapshot.get("model_versions", []))} entri')
pd.DataFrame(batch_windows)

## 5. Validasi Input dan Parameter

Pipeline harus aman dari input tidak valid. Cell ini memverifikasi semua prasyarat sebelum pipeline dijalankan.

In [ ]:
assert len(history) > 0, 'Processed history must not be empty.'
assert cp3_config['batch_count'] >= 2, 'Batch count must be at least 2.'
assert 0 < cp3_config['drift_threshold'] < 1, 'Drift threshold must be a ratio between 0 and 1.'
assert 0 < cp3_config['quality_gate'] < 1, 'Quality gate must be a ratio between 0 and 1.'
assert len(batches) >= 2, 'Temporal split must create at least two batches.'

print('Semua validasi input CP3 berhasil.')

## 6. Fungsi Inti CP3

Fungsi inti sudah diimplementasikan di `app/continual_learning.py`. Notebook ini memanggil pipeline tersebut
agar alur retraining, drift detection, dan checkpointing bisa dilihat end-to-end.

Alur yang terjadi di dalam `run_cp3_pipeline`:
1. Load data → split temporal → holdout = batch terakhir
2. Untuk tiap batch training: evaluasi model saat ini → hitung MAE
3. Jika MAE naik > `drift_threshold` dari referensi → **drift terdeteksi**
4. Jika drift → latih kandidat dari semua data yang sudah terlihat
5. Evaluasi kandidat di holdout → bandingkan dengan model saat ini
6. Kandidat diterima hanya jika holdout MAE turun ≥ `quality_gate` dan RMSE tidak memburuk

## 7. Jalankan Pipeline CP3 End-to-End

In [ ]:
pipeline_summary = run_cp3_pipeline(
    max_rows=cp3_config['max_rows_demo'],
    batch_count=cp3_config['batch_count'],
    drift_threshold=cp3_config['drift_threshold'],
    quality_gate=cp3_config['quality_gate'],
    persist=cp3_config['persist_outputs'],
)

print('Pipeline selesai.')
print(f"Total batch         : {pipeline_summary['batch_count']}")
print(f"Drift events        : {len(pipeline_summary['drift_events'])}")
print(f"Checkpoint diterima : {len(pipeline_summary['promoted_versions'])}")
print(f"Checkpoint ditolak  : {len(pipeline_summary['rejected_candidates'])}")

## 8. Verifikasi Versi dan Drift Events

In [ ]:
initial_versions = read_model_versions()
final_versions = pipeline_summary['promoted_versions']
rejected_versions = pipeline_summary['rejected_candidates']

print(f'Versi di model_versions.json : {len(initial_versions.get("model_versions", []))}')
print(f'Dipromosikan sesi ini        : {len(final_versions)}')
print(f'Ditolak sesi ini             : {len(rejected_versions)}')

if pipeline_summary['drift_events']:
    print('\nDrift events:')
    display(pd.DataFrame(pipeline_summary['drift_events']))

## 9. Uji Edge Case: Logika detect_drift

In [ ]:
# Kasus jelas: MAE naik >15%
simple_drift = detect_drift(10.0, 12.0, threshold=cp3_config['drift_threshold'])
# Kasus batas bawah: MAE naik tepat 10% (di bawah threshold 15%)
edge_drift = detect_drift(10.0, 11.0, threshold=cp3_config['drift_threshold'])

assert simple_drift is True,  f'Seharusnya drift (12 > 10*1.15=11.5): {simple_drift}'
assert edge_drift is False, f'Seharusnya tidak drift (11 < 10*1.15=11.5): {edge_drift}'

print({'simple_drift (12 vs ref 10)': simple_drift, 'edge_drift (11 vs ref 10)': edge_drift})
print('Semua edge case detect_drift lulus.')

## 10. Ringkasan Hasil dan Simpan Output

Tampilkan ringkasan tabel batch dan simpan summary ke JSON.

In [ ]:
batch_summary = pd.DataFrame(pipeline_summary['batches'])
drift_summary = pd.DataFrame(pipeline_summary['drift_events']) if pipeline_summary['drift_events'] else pd.DataFrame()
version_summary = pd.DataFrame(read_model_versions().get('model_versions', []))

print('Batch summary:')
display(batch_summary)

if not drift_summary.empty:
    print('\nDrift events:')
    display(drift_summary)

print('\nVersion registry:')
if not version_summary.empty:
    display(version_summary[['version', 'accepted', 'artifact_path', 'created_at']])
else:
    print('  (kosong — persist=False, tidak ada versi baru yang disimpan)')

# Simpan summary ke file
summary_path = OUTPUT_DIR / 'cp3_summary.json'
with summary_path.open('w', encoding='utf-8') as handle:
    json.dump(pipeline_summary, handle, indent=2, ensure_ascii=False, default=str)

print(f'\nSummary disimpan ke: {summary_path}')
print('CP3 implementation summary:')
print('1. Data processed dibagi menjadi batch temporal.')
print('2. Drift dideteksi dari kenaikan relatif MAE antar batch.')
print('3. Candidate model hanya dipromosikan jika lolos quality gate holdout.')
print('4. Versioning model dicatat di model_versions.json.')